In [2]:
!pip install langchain langchain-core langchain-community langchain-huggingface chromadb sentence-transformers langchain-google-genai

In [3]:
!pip install pdfplumber

import pdfplumber
import re
import pandas as pd # 引入 pandas 方便我們用表格來檢視分塊成果

def process_insurance_pdf(file_path, company_name):
    full_text = ""
    print(f"開始解析 {company_name} 旅平險保單...")

    # 2. 讀取 PDF 並萃取文字
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"
    except FileNotFoundError:
        return f"錯誤：找不到 {file_path}，請確認是否已上傳至 Colab！"

    # 3. 資料清洗 (Data Cleaning)
    if company_name == "國泰":
        # 濾除國泰保單特有的巨大「樣張」浮水印干擾文字
        full_text = full_text.replace("樣張", "")

    # 去除多餘的空白行，保持段落連貫
    full_text = re.sub(r'\n+', '\n', full_text)

    # 4. 智能語意分塊 (Semantic Chunking)
    # 使用正規表達式匹配「第一條」、「第二十條」等作為切塊錨點
    chunks = re.split(r'(第[一二三四五六七八九十百]+條\s+.*?\n)', full_text)

    processed_chunks = []
    # chunks 的第 0 個元素通常是保單開頭的說明，我們從第 1 個元素開始抓取條款
    for i in range(1, len(chunks), 2):
        clause_title = chunks[i].strip()
        # 確保不會發生 index out of range
        clause_content = chunks[i+1].strip() if (i+1) < len(chunks) else ""

        chunk_data = {
            "保險公司": company_name,
            "條款標題": clause_title,
            # 將公司、標題與內文組合成完整區塊給 AI 讀取
            "完整文本 (Chunk)": f"[{company_name}] {clause_title}\n{clause_content}"
        }
        processed_chunks.append(chunk_data)

    return processed_chunks

# 5. 執行處理
nanshan_chunks = process_insurance_pdf("南山產物旅平險保單條款.pdf", "南山")
cathay_chunks = process_insurance_pdf("國泰旅平險保單條款.pdf", "國泰")

# 6. 將結果轉換為 DataFrame 方便檢視與後續處理
if isinstance(nanshan_chunks, list) and isinstance(cathay_chunks, list):
    all_chunks = nanshan_chunks + cathay_chunks
    df_chunks = pd.DataFrame(all_chunks)

    print(f"\n解析完成！共切出 {len(all_chunks)} 個條款區塊。")
    # 顯示前 5 筆分塊結果讓你看一下效果
    display(df_chunks.head())
else:
    print(nanshan_chunks) # 印出錯誤訊息

開始解析 南山 旅平險保單...
開始解析 國泰 旅平險保單...

解析完成！共切出 168 個條款區塊。


,保險公司,條款標題,完整文本 (Chunk)
0,南山,第六條 失能保險金的給付,[南山] 第六條 失能保險金的給付\n108.12.13南山保字第1080001540號函備...
1,南山,第一條 保險契約之構成,[南山] 第一條 保險契約之構成\n一者，本公司給付失能保險金，其金額按該表所列之給付比例計...
2,南山,第二條 承保範圍,[南山] 第二條 承保範圍\n屬失能等級不同時，給付較嚴重項目的失能保險金。\n被保險人於本...
3,南山,第三條 保險期間的始日與終日 前項情形，若被保險人扣除以前的失能後得領取之保險金低於單獨請,[南山] 第三條 保險期間的始日與終日 前項情形，若被保險人扣除以前的失能後得領取之保險金低...
4,南山,第四條 保險期間的延長 金時，本公司累計給付金額最高以保險金額為限。,[南山] 第四條 保險期間的延長 金時，本公司累計給付金額最高以保險金額為限。\n如被保險人...


In [4]:
!pip install langchain langchain-core langchain-community langchain-huggingface chromadb sentence-transformers

In [5]:
# 載入套件 (修正了 Document 的載入路徑)
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("套件載入完成，準備建立資料庫...")

# 將第一步切好的 chunks 轉換為 LangChain 的 Document 格式
documents = []
for chunk in all_chunks: # 這裡會接續使用你第一步跑出來的 all_chunks
    # 保留 metadata，這對於未來追蹤答案來源 (Citations) 非常重要
    doc = Document(
        page_content=chunk["完整文本 (Chunk)"],
        metadata={
            "保險公司": chunk["保險公司"],
            "條款標題": chunk["條款標題"]
        }
    )
    documents.append(doc)

# 初始化 Embedding 模型
# 選用支援中文的開源模型 shibing624/text2vec-base-chinese
print("正在下載並載入語意向量 (Embedding) 模型，這可能需要一兩分鐘...")
embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")

# 建立 Chroma 向量資料庫並存入資料
print("正在將保險條款轉換為數學向量，並寫入 ChromaDB 資料庫...")
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./insurance_chroma_db" # 將資料庫存在 Colab 的暫存資料夾中
)

print(f"\n🎉 成功建立向量資料庫！共存入 {vector_db._collection.count()} 筆條款區塊。")

# 實地測試檢索演算法 (Retrieval)
# 測試你的系統是否能捕捉到語意
test_query = "如果我的飛機延誤了五個小時，有理賠嗎？"
print(f"\n🔍 測試檢索問題: '{test_query}'")

# 請資料庫根據語意向量找出最相關的 3 個條款 (k=8)
results = vector_db.similarity_search(test_query, k=8)

for i, res in enumerate(results):
    print(f"\n--- 相關條款 {i+1} ---")
    print(f"來源: {res.metadata['保險公司']} - {res.metadata['條款標題']}")
    # 印出前 150 個字預覽
    print(f"內容: {res.page_content[:150]}...")

套件載入完成，準備建立資料庫...
正在下載並載入語意向量 (Embedding) 模型，這可能需要一兩分鐘...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


正在將保險條款轉換為數學向量，並寫入 ChromaDB 資料庫...

🎉 成功建立向量資料庫！共存入 336 筆條款區塊。

🔍 測試檢索問題: '如果我的飛機延誤了五個小時，有理賠嗎？'

--- 相關條款 1 ---
來源: 南山 - 第二十一條 承保範圍 第二十五條 特別不保事項
內容: [南山] 第二十一條 承保範圍 第二十五條 特別不保事項
被保險人於本保險契約保險期間內，以乘客身分預定搭乘之定期航班 對於下列事項或該事項所致之損失，本公司不負理賠責任：
發生延誤，致被保險人實際出發時間較預定出發時間延誤四小時以上 一、因旅行社或公共交通工具業者破產、清算或債務不履行。
者，本公...

--- 相關條款 2 ---
來源: 南山 - 第二十一條 承保範圍 第二十五條 特別不保事項
內容: [南山] 第二十一條 承保範圍 第二十五條 特別不保事項
被保險人於本保險契約保險期間內，以乘客身分預定搭乘之定期航班 對於下列事項或該事項所致之損失，本公司不負理賠責任：
發生延誤，致被保險人實際出發時間較預定出發時間延誤四小時以上 一、因旅行社或公共交通工具業者破產、清算或債務不履行。
者，本公...

--- 相關條款 3 ---
來源: 南山 - 第二十六條 理賠文件
內容: [南山] 第二十六條 理賠文件
接班機實際出發之時止。
被保險人向本公司申請理賠時，應檢具下列文件：
下列情形視為同一事故，以給付一次為限：
一、共同文件：
一、去程於同一機場所發生之班機延誤或班機取消。
(一)理賠申請書。
二、回程於同一機場所發生之班機延誤或班機取消。
(二)費用單據正本。
三、...

--- 相關條款 4 ---
來源: 南山 - 第二十六條 理賠文件
內容: [南山] 第二十六條 理賠文件
接班機實際出發之時止。
被保險人向本公司申請理賠時，應檢具下列文件：
下列情形視為同一事故，以給付一次為限：
一、共同文件：
一、去程於同一機場所發生之班機延誤或班機取消。
(一)理賠申請書。
二、回程於同一機場所發生之班機延誤或班機取消。
(二)費用單據正本。
三、...

--- 相關條款 5 ---
來源: 南山 - 第二十條 理賠文件 六、被保險人自行安排替代班機之目的地與原預定搭乘班機非屬同一
內容: [南山] 第二十條 理賠文件 六、被保險人自行安排替代班機之目的

In [8]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")


✅ AI 旅平險專員已上線！
💡 (輸入 'q' 即可結束)

👤 你的問題：申領「意外身故保險金」時，受益人需要提供哪些文件？

=============== 提問確認 ===============
  申領「意外身故保險金」時，受益人需要提供哪些文件？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：

根據目前條款無法得知申領「意外身故保險金」時，受益人需要提供哪些文件。

參考資料中，[資料 1]、[資料 2]、[資料 4]、[資料 8] 提及的是「傷害醫療保險金」的申領文件；[資料 3]
提及的是「意外失能保險金」、「水陸大眾運輸交通意外失能保險金」或「航空大眾運輸交通意外失能保險金」的申領文件。

[資料 6] 南山-第四條 理賠文件 僅提供一般理賠申請書及被保險人身分證明文件，並未特別指明為「意外身故保險金」所需文件。

[資料 7] 國泰-第二十三條 受益人的指定及變更
雖然提及「意外身故」給付，但主要說明受益人的指定與變更，並未列出申領「意外身故保險金」時所需的文件。

----------------------------------------
📚 參考來源：
• 國泰 : 第三條 傷害醫療保險金的申領
• 南山 : 第二條 傷害醫療保險金的申領 退還要保人。
• 國泰 : 第二十一條 意外失能保險金、水陸大眾運輸交通意外失能保險金、航空大眾運輸交通意外失能保險金的申領
• 南山 : 第十二條 傷害醫療保險金的申領
• 國泰 : 第十八條 保險事故的通知與保險金的申請時間
• 南山 : 第四條 理賠文件 第三條 事故發生後之處理
• 國泰 : 第二十三條 受益人的指定及變更
• 南山 : 第四條 保險期間的始日與終日 一、保險金申請書。
----------------------------------------

👤 你的問題：q


In [ ]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")

In [7]:
# === 最終修正版：輸入與回答雙向分行 ===

import textwrap

print("\n✅ AI 旅平險專員已上線！")
print("💡 (輸入 'q' 即可結束)\n")

while True:
    # 這裡讓使用者輸入
    raw_input = input("👤 你的問題：")

    if raw_input.lower() in ['q', '退出']:
        break
    if not raw_input.strip():
        continue

    # --- 修正點 1：讓「你的問題」在顯示時也分行 ---
    # 我們重複印一次經過處理的問題，確保長度適中
    print("\n" + "="*15 + " 提問確認 " + "="*15)
    for line in textwrap.wrap(raw_input, width=70):
        print(f"  {line}")
    print("="*40)

    print("\n⏳ 系統檢索與思考中...\n")

    # 檢索與處理
    retrieved_docs = vector_db.similarity_search(raw_input, k=8)
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}] {doc.metadata['保險公司']}-{doc.metadata['條款標題']}\n{doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    # 呼叫 LLM
    final_prompt = PROMPT.format(context=context_text, question=raw_input)
    response = llm.invoke(final_prompt)

    # --- 修正點 2：控制 AI 回答寬度並增加段落感 ---
    print("🤖 AI 專員回答：\n")

    # 這裡先處理段落換行，再處理寬度，確保內容不會擠成一坨
    paragraphs = response.content.split('\n')
    for para in paragraphs:
        if para.strip():
            # width=70 是最安全的中文字顯示寬度，不會觸發捲軸
            print(textwrap.fill(para, width=70))
            print() # 每個段落後多空一行，視覺更舒服

    # 縮短分隔線，防止 image_9f3ee3.png 的問題再次發生
    short_line = "-" * 40
    print(short_line)
    print("📚 參考來源：")
    for doc in unique_docs:
        print(f"• {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print(short_line + "\n")


✅ AI 旅平險專員已上線！
💡 (輸入 'q' 即可結束)

👤 你的問題：申領「意外身故保險金」時，受益人需要提供哪些文件？

=============== 提問確認 ===============
  申領「意外身故保險金」時，受益人需要提供哪些文件？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：

根據目前條款無法得知申領「意外身故保險金」時，受益人需要提供哪些文件。

參考資料中，[資料 1]、[資料 2]、[資料 4]、[資料 8] 提及的是「傷害醫療保險金」的申領文件；[資料 3]
提及的是「意外失能保險金」、「水陸大眾運輸交通意外失能保險金」或「航空大眾運輸交通意外失能保險金」的申領文件。

[資料 6] 南山-第四條 理賠文件 僅提供一般理賠申請書及被保險人身分證明文件，並未特別指明為「意外身故保險金」所需文件。

[資料 7] 國泰-第二十三條 受益人的指定及變更
雖然提及「意外身故」給付，但主要說明受益人的指定與變更，並未列出申領「意外身故保險金」時所需的文件。

----------------------------------------
📚 參考來源：
• 國泰 : 第三條 傷害醫療保險金的申領
• 南山 : 第二條 傷害醫療保險金的申領 退還要保人。
• 國泰 : 第二十一條 意外失能保險金、水陸大眾運輸交通意外失能保險金、航空大眾運輸交通意外失能保險金的申領
• 南山 : 第十二條 傷害醫療保險金的申領
• 國泰 : 第十八條 保險事故的通知與保險金的申請時間
• 南山 : 第四條 理賠文件 第三條 事故發生後之處理
• 國泰 : 第二十三條 受益人的指定及變更
• 南山 : 第四條 保險期間的始日與終日 一、保險金申請書。
----------------------------------------



KeyboardInterrupt: Interrupted by user

In [10]:
import os
import re
import pdfplumber
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import google.generativeai as genai

# ===  設定 API Key 與動態選擇模型 ===
API_KEY = "YOUR API"
os.environ["GOOGLE_API_KEY"] = API_KEY
genai.configure(api_key=API_KEY)

available_models = [m.name.replace("models/", "") for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
chosen_model = next((m for m in available_models if "pro" in m or "flash" in m), available_models[-1] if available_models else None)

# ===  PDF 解析與清洗函數 ===
def process_insurance_pdf(file_path, company_name):
    full_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text: full_text += text + "\n"
    if company_name == "國泰":
        full_text = full_text.replace("樣張", "")
    full_text = re.sub(r'\n+', '\n', full_text)
    chunks = re.split(r'(第[一二三四五六七八九十百]+條\s+.*?\n)', full_text)

    return [
        Document(
            page_content=f"[{company_name}] {chunks[i].strip()}\n{chunks[i+1].strip() if (i+1) < len(chunks) else ''}",
            metadata={"保險公司": company_name, "條款標題": chunks[i].strip()}
        ) for i in range(1, len(chunks), 2)
    ]

# ===  重新建立唯一且乾淨的資料庫 ===
print("📚 正在重新解析 PDF 並建立乾淨的資料庫 (約需1分鐘)...")
documents = process_insurance_pdf("南山產物旅平險保單條款.pdf", "南山") + \
            process_insurance_pdf("國泰旅平險保單條款.pdf", "國泰")

embeddings = HuggingFaceEmbeddings(model_name="shibing624/text2vec-base-chinese")
# 注意這裡使用 from_documents 重新建立資料庫！
vector_db = Chroma.from_documents(documents=documents, embedding=embeddings)

# ===  初始化 RAG 提示詞 ===
llm = ChatGoogleGenerativeAI(model=chosen_model, temperature=0)
prompt_template = """
你是一位專業的旅平險 AI 專員。請根據以下提供的【保險條款參考資料】來回答客戶的問題。
【嚴格要求】
1. 你的回答必須完全基於參考資料，絕不能自己瞎編。
2. 回答時，必須明確標註來源！例如：「根據 [保險公司] - [條款標題] 的規定...」。
3. 如果參考資料中沒有答案，請誠實回答「根據目前條款無法得知」。

【保險條款參考資料】
{context}

客戶問題：{question}
專業回答：
"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# ===  啟動對話迴圈 (參數已調整為 k=8 解決單一公司問題) ===
print("\n✅ AI 旅平險專員已重新上線！(資料庫已更新完畢)")
print("💡 (在下方輸入框打字，輸入 'q' 即可結束對話)\n")

while True:
    user_input = input("👤 你的問題：")
    if user_input.lower() in ['q', '退出', 'quit', 'exit']:
        print("\n👋 結束對話，AI 專員下線囉！")
        break
    if not user_input.strip(): continue

    print("\n⏳ 系統檢索與思考中...\n")
    retrieved_docs = vector_db.similarity_search(user_input, k=8)

    # 過濾重複抓取的文獻，確保不會印出雙胞胎
    seen_titles = set()
    unique_docs = []
    for doc in retrieved_docs:
        title = f"{doc.metadata['保險公司']} - {doc.metadata['條款標題']}"
        if title not in seen_titles:
            seen_titles.add(title)
            unique_docs.append(doc)

    context_text = "".join([f"\n[資料 {i+1}]\n來源: {doc.metadata['保險公司']} - {doc.metadata['條款標題']}\n內容: {doc.page_content}\n" for i, doc in enumerate(unique_docs)])

    response = llm.invoke(PROMPT.format(context=context_text, question=user_input))

    print("🤖 AI 專員回答：")
    print(response.content)
    print("\n--------------------------------------------------")
    print("📚 本次回答參考的唯一底層文獻：")
    for doc in unique_docs:
        print(f"- {doc.metadata['保險公司']} : {doc.metadata['條款標題']}")
    print("==================================================\n")

📚 正在重新解析 PDF 並建立乾淨的資料庫 (約需1分鐘)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ AI 旅平險專員已重新上線！(資料庫已更新完畢)
💡 (在下方輸入框打字，輸入 'q' 即可結束對話)

👤 你的問題：位 12 歲少年投保國泰旅平險，保額為 100 萬。他在搭機返國時因空難不幸身故。請問根據條款，受益人能領取的保險金項目為何？金額如何計算？

⏳ 系統檢索與思考中...

🤖 AI 專員回答：
您好，根據您提供的條款資料，針對 12 歲少年投保國泰旅平險，保額 100 萬，因空難身故的情況，專業回答如下：

1.  **航空大眾運輸交通意外身故保險金：**
    根據 [國泰 - 第八條 航空大眾運輸交通意外身故保險金或喪葬費用保險金的給付] 的規定：「訂立本契約時，以未滿十五足歲之未成年人為被保險人，其『航空大眾運輸交通意外身故保險金』之給付於被保險人滿十五足歲之日起發生效力。」
    由於被保險人為 12 歲，未滿十五足歲，因此「航空大眾運輸交通意外身故保險金」的給付尚未發生效力，受益人無法領取此項保險金。

2.  **意外身故保險金：**
    [國泰 - 第八條 航空大眾運輸交通意外身故保險金或喪葬費用保險金的給付] 中提到：「本公司除給付『意外身故保險金』外，另按本契約保險單上所記載之保險金額的百分之二十五，給付『航空大眾運輸交通意外身故保險金』。」
    此條款提及「意外身故保險金」為一獨立項目，但提供的參考資料中，並未包含針對未滿十五足歲之未成年人，在航空交通意外身故情況下，「意外身故保險金」的具體給付條件、金額計算方式或是否有其他限制的條款。

**結論：**
根據目前提供的【保險條款參考資料】：
*   **保險金項目：** 「航空大眾運輸交通意外身故保險金」因被保險人未滿十五足歲，故不予給付。
*   **金額計算：** 關於「意外身故保險金」的給付條件與金額計算，根據目前條款無法得知。

--------------------------------------------------
📚 本次回答參考的唯一底層文獻：
- 國泰 : 第十一條 航空大眾運輸交通意外失能保險金的給付
- 國泰 : 第八條 航空大眾運輸交通意外身故保險金或喪葬費用保險金的給付
- 國泰 : 第二十一條 意外失能保險金、水陸大眾運輸交通意外失能保險金、航空大眾運輸交通意外失能保險金的申領

👤 你的問題：被保險人飲酒後駕車發生事故